# Local Result Summary: `results/feature_ablation`

Notebook này chỉ đọc kết quả nằm trong `results/feature_ablation` và ghi bảng/plot vào `_summary/`.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path("/tmp") / "matplotlib"))

import matplotlib
if "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    display
except NameError:
    def display(value):
        print(value)

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name in {"notebook", "notebooks"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
SUMMARY_DIR = (PROJECT_ROOT / "results/feature_ablation").resolve()
OUT_DIR = SUMMARY_DIR / "_summary"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["WA", "UA", "WF1", "Macro-F1"]
PRIMARY_METRIC = "Macro-F1"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("SUMMARY_DIR  =", SUMMARY_DIR)
print("OUT_DIR      =", OUT_DIR)


## 2. Helpers


In [ ]:
def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def pct(value) -> float:
    if pd.isna(value):
        return np.nan
    return 100.0 * float(value)


def mean_std_text(mean, std) -> str:
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{pct(mean):.2f}"
    return f"{pct(mean):.2f} ± {pct(std):.2f}"


def relative_parts(path: Path) -> tuple[str, ...]:
    try:
        return path.relative_to(SUMMARY_DIR).parts
    except ValueError:
        return path.parts


def variant_from_cross_summary(path: Path) -> str:
    parts = list(relative_parts(path))
    if "cross_session" in parts:
        idx = parts.index("cross_session")
        if idx == 0:
            return path.parent.parent.parent.name
        return "/".join(parts[:idx])
    return path.parent.name


def variant_from_metrics(path: Path) -> str:
    parts = list(relative_parts(path))
    if "cross_session" in parts:
        idx = parts.index("cross_session")
        return "/".join(parts[:idx]) if idx > 0 else path.parent.parent.parent.name
    if "phase_tests" in parts:
        idx = parts.index("phase_tests")
        return "/".join(parts[:idx])
    return "/".join(parts[:-1]) if len(parts) > 1 else path.parent.name


def run_id_from_path(path: Path) -> str:
    parts = relative_parts(path)
    for part in parts:
        if part.startswith("run_"):
            return part
    return "single_run"


def test_session_from_path(path: Path):
    match = re.search(r"test_Ses(\d+)", str(path))
    return int(match.group(1)) if match else np.nan


def flatten_metric_payload(payload: dict) -> dict:
    row = {}
    source = payload.get("test", payload)
    for metric in METRICS:
        value = source.get(metric, payload.get(metric))
        row[metric] = float(value) if value is not None else np.nan
    row["best_epoch"] = payload.get("best_epoch", payload.get("checkpoint_epoch", np.nan))
    return row


## 3. Cross-Session Aggregate


In [ ]:
cross_rows = []
fold_rows = []
for path in sorted(SUMMARY_DIR.rglob("cross_session_summary.json")):
    payload = read_json(path)
    variant = variant_from_cross_summary(path)
    run_id = payload.get("run_name") or run_id_from_path(path)
    row = {
        "variant": variant,
        "run_id": run_id,
        "trainer_module": payload.get("trainer_module", ""),
        "summary_path": str(path.relative_to(PROJECT_ROOT)),
    }
    aggregate = payload.get("aggregate", {})
    for metric in METRICS:
        stats = aggregate.get(metric, {}) if isinstance(aggregate, dict) else {}
        row[f"{metric}_mean"] = float(stats.get("mean", np.nan))
        row[f"{metric}_std"] = float(stats.get("std", np.nan))
        row[f"{metric}_n"] = int(stats.get("n", 0)) if not pd.isna(stats.get("n", np.nan)) else 0
    cross_rows.append(row)

    for fold in payload.get("folds", []):
        metrics = fold.get("metrics", {})
        fold_row = {
            "variant": variant,
            "run_id": run_id,
            "test_session": int(fold.get("test_session", test_session_from_path(Path(str(fold.get("output_dir", "")))) or -1)),
            "output_dir": fold.get("output_dir", ""),
            "summary_path": str(path.relative_to(PROJECT_ROOT)),
        }
        for metric in METRICS:
            fold_row[metric] = float(metrics.get(metric, np.nan))
        fold_rows.append(fold_row)

aggregate_df = pd.DataFrame(cross_rows)
folds_df = pd.DataFrame(fold_rows)

if not aggregate_df.empty:
    aggregate_df = aggregate_df.sort_values(f"{PRIMARY_METRIC}_mean", ascending=False).reset_index(drop=True)
    aggregate_df.to_csv(OUT_DIR / "cross_session_aggregate.csv", index=False)
if not folds_df.empty:
    folds_df = folds_df.sort_values(["variant", "run_id", "test_session"]).reset_index(drop=True)
    folds_df.to_csv(OUT_DIR / "cross_session_folds.csv", index=False)

print(f"Found {len(aggregate_df)} cross-session summaries and {len(folds_df)} folds")
if aggregate_df.empty:
    print("No cross_session_summary.json files found under", SUMMARY_DIR)
else:
    display(aggregate_df)


## 4. Mean ± Std Table


In [ ]:
if aggregate_df.empty:
    formatted_df = pd.DataFrame()
else:
    formatted_rows = []
    for _, row in aggregate_df.iterrows():
        item = {
            "variant": row["variant"],
            "run_id": row["run_id"],
            "trainer_module": row.get("trainer_module", ""),
        }
        for metric in METRICS:
            item[metric] = mean_std_text(row.get(f"{metric}_mean", np.nan), row.get(f"{metric}_std", np.nan))
        formatted_rows.append(item)
    formatted_df = pd.DataFrame(formatted_rows)
    formatted_df.to_csv(OUT_DIR / "cross_session_aggregate_formatted.csv", index=False)
    display(formatted_df)


## 5. Session Heatmap


In [ ]:
if folds_df.empty:
    print("No fold-level data for heatmap.")
else:
    pivot = folds_df.pivot_table(
        index="variant",
        columns="test_session",
        values=PRIMARY_METRIC,
        aggfunc="max",
    )
    pivot = pivot.reindex(aggregate_df["variant"].drop_duplicates().tolist())
    plt.figure(figsize=(max(7, 1.0 * pivot.shape[1] + 4), max(3.5, 0.45 * len(pivot) + 1.5)))
    sns.heatmap(pivot * 100.0, annot=True, fmt=".2f", cmap="mako", cbar_kws={"label": f"{PRIMARY_METRIC} (%)"})
    plt.title(f"{PRIMARY_METRIC} by held-out session")
    plt.xlabel("Held-out session")
    plt.ylabel("")
    plt.tight_layout()
    fig_path = FIG_DIR / "cross_session_heatmap.png"
    plt.savefig(fig_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Saved", fig_path)


## 6. Standalone Metrics


In [ ]:
single_rows = []
for path in sorted(SUMMARY_DIR.rglob("metrics.json")):
    parts = set(relative_parts(path))
    if "phase_tests" in parts:
        continue
    if "cross_session" in parts:
        # Fold metrics are already represented by cross_session_summary.json.
        continue
    payload = read_json(path)
    row = {
        "variant": variant_from_metrics(path),
        "run_id": run_id_from_path(path),
        "metrics_path": str(path.relative_to(PROJECT_ROOT)),
    }
    row.update(flatten_metric_payload(payload))
    single_rows.append(row)

single_metrics_df = pd.DataFrame(single_rows)
if not single_metrics_df.empty:
    single_metrics_df = single_metrics_df.sort_values(PRIMARY_METRIC, ascending=False).reset_index(drop=True)
    single_metrics_df.to_csv(OUT_DIR / "single_run_metrics.csv", index=False)
    display(single_metrics_df)
else:
    print("No standalone metrics.json files found outside cross_session folds.")


## 7. Phase Test Metrics


In [ ]:
phase_rows = []
for path in sorted(SUMMARY_DIR.rglob("phase_test_metrics.csv")):
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        print("Failed to read", path, exc)
        continue
    variant = variant_from_metrics(path)
    run_id = run_id_from_path(path)
    for _, row in df.iterrows():
        item = row.to_dict()
        item.update({
            "variant": variant,
            "run_id": run_id,
            "phase_metrics_path": str(path.relative_to(PROJECT_ROOT)),
        })
        phase_rows.append(item)

phase_df = pd.DataFrame(phase_rows)
if not phase_df.empty:
    metric_col = "test_Macro-F1" if "test_Macro-F1" in phase_df.columns else None
    if metric_col:
        phase_df = phase_df.sort_values(metric_col, ascending=False).reset_index(drop=True)
    phase_df.to_csv(OUT_DIR / "phase_test_metrics.csv", index=False)
    display(phase_df)
else:
    print("No phase_test_metrics.csv files found.")


## 8. Prediction Files


In [ ]:
prediction_rows = []
for path in sorted(SUMMARY_DIR.rglob("predictions.csv")):
    prediction_rows.append({
        "variant": variant_from_metrics(path),
        "run_id": run_id_from_path(path),
        "test_session": test_session_from_path(path),
        "prediction_path": str(path.relative_to(PROJECT_ROOT)),
    })

prediction_df = pd.DataFrame(prediction_rows)
if not prediction_df.empty:
    prediction_df.to_csv(OUT_DIR / "prediction_file_index.csv", index=False)
    display(prediction_df.head(50))
    print(f"Prediction files: {len(prediction_df)}")
else:
    print("No predictions.csv files found.")


## 9. Report


In [ ]:
report = [
    f"# Local Result Summary: `{SUMMARY_DIR.relative_to(PROJECT_ROOT)}`",
    "",
    f"- Cross-session summaries: {len(aggregate_df)}",
    f"- Cross-session folds: {len(folds_df)}",
    f"- Standalone metric files: {len(single_metrics_df)}",
    f"- Phase metric rows: {len(phase_df)}",
    f"- Prediction files: {len(prediction_df)}",
    "",
]
if not formatted_df.empty:
    report.extend(["## Cross-Session Aggregate", "", formatted_df.to_markdown(index=False), ""])
if not single_metrics_df.empty:
    report.extend(["## Standalone Metrics", "", single_metrics_df.to_markdown(index=False), ""])
if not phase_df.empty:
    report.extend(["## Phase Test Metrics", "", phase_df.to_markdown(index=False), ""])

report_path = OUT_DIR / "summary.md"
report_path.write_text("\n".join(report) + "\n", encoding="utf-8")
print("Saved", report_path)


## 10. Saved Outputs


In [ ]:
outputs = sorted(p.relative_to(OUT_DIR) for p in OUT_DIR.rglob("*") if p.is_file())
print(f"Saved {len(outputs)} files under {OUT_DIR}")
for path in outputs:
    print(path)
